# M04-L06 — Transfer Learning, Dropout e Funções de Perda
## Laboratório do participante

Nesta lição, vamos comparar **quatro regimes** da mesma ResNet18:

1. **Scratch** — `weights=None`, tudo treinável;
2. **FT classificador** — pesos pré-treinados, backbone congelado, só a cabeça treinável;
3. **FT parcial** — pesos pré-treinados, `layer4 + cabeça` treináveis;
4. **FT completo** — pesos pré-treinados, toda a rede treinável com atualizações mais cuidadosas.

![Quatro regimes](https://drive.google.com/thumbnail?id=1NhOjqm0dweDbvJaMNwPKEmTOp-cA88v0&sz=w1600)

> O experimento “pré-treinado totalmente destravado com os mesmos hiperparâmetros do scratch” aparece como **diagnóstico de uso ingênuo do FT completo**. Ele não é uma quinta versão.

**Esta é a última lição do M04.**

**Regras do laboratório:** decisões são tomadas pela validação; `dataset_iris/test` só será usado quando a configuração final já estiver escolhida.

## 1. Retomada da M04-L05 e preparação do ambiente

Reutilize a mesma pasta `dataset_iris` e o arquivo `resultados_modulo_m04.csv` salvos em `Meu Drive/dados_modulo_04/` nas lições anteriores. Monte o mesmo Drive na célula seguinte; se algum item estiver ausente, envie `dataset_iris.zip` (com a pasta `dataset_iris` dentro) ou o CSV da L05 quando solicitado.

Na L05, você registrou quais categorias de augmentation pareciam plausíveis. Na célula abaixo, ajuste **apenas os três booleanos** para reproduzir sua escolha. Os valores iniciais (`True, True, False`) são apenas um exemplo executável.

In [ ]:
from pathlib import Path
import random
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive, files
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

SEMENTE = 42
random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEMENTE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

drive.mount('/content/drive')
pasta_modulo_drive = Path('/content/drive/MyDrive/dados_modulo_04')
base_dir = pasta_modulo_drive / 'dataset_iris'
if not base_dir.is_dir():
    print('dataset_iris não encontrado no Drive. Envie dataset_iris.zip da M04-L03.')
    enviados = files.upload()
    if 'dataset_iris.zip' not in enviados:
        raise FileNotFoundError('Envie dataset_iris.zip contendo train, val e test.')
    with zipfile.ZipFile('/content/dataset_iris.zip') as arquivo_zip:
        if any(nome.startswith('/') or '..' in Path(nome).parts for nome in arquivo_zip.namelist()):
            raise ValueError('O ZIP contém caminhos inválidos.')
        arquivo_zip.extractall('/content')
    base_dir = Path('/content/dataset_iris')

classes_esperadas = {'setosa', 'versicolor', 'virginica'}
for particao in ('train', 'val', 'test'):
    pasta = base_dir / particao
    if not pasta.is_dir() or {p.name for p in pasta.iterdir() if p.is_dir()} != classes_esperadas:
        raise FileNotFoundError(f'Confira as partições e as três classes em {base_dir}.')

TAMANHO_IMAGEM = (256, 256)
BATCH_SIZE = 16

# COPIE a decisão que você tomou na M04-L05.
USAR_GEOMETRICO = True
USAR_COR = True
USAR_DISTORCAO = False

arquivo_resultados = pasta_modulo_drive / 'resultados_modulo_m04.csv'
if not arquivo_resultados.is_file():
    print('CSV da L05 não encontrado no Drive. Envie resultados_modulo_m04.csv.')
    enviados = files.upload()
    if 'resultados_modulo_m04.csv' not in enviados:
        raise FileNotFoundError('Envie resultados_modulo_m04.csv da M04-L05.')
    arquivo_resultados = Path('/content/resultados_modulo_m04.csv')
resultados_modulo = pd.read_csv(arquivo_resultados)
if not {'modelo', 'acuracia_teste'}.issubset(resultados_modulo.columns):
    raise ValueError('O CSV precisa conter modelo e acuracia_teste.')
if not resultados_modulo['modelo'].astype(str).str.startswith('ResNet18IrisMelhorada_L05_').any():
    raise ValueError('O CSV precisa conter a linha da ResNet18 melhorada na M04-L05.')
display(resultados_modulo)


## 2. Normalização associada aos pesos pré-treinados

Na L05 usamos `Normalize` com valores didáticos. Agora os pesos pré-treinados fornecem as estatísticas esperadas pelo pré-processamento.

![Normalização do pré-treino](https://drive.google.com/thumbnail?id=1PDrc0-mR9AO_Ncbi4SuIqh5I69rblmyR&sz=w1600)

Nesta última lição do módulo, trabalharemos com imagens em `256×256`, a resolução original informada para o dataset. O mesmo tamanho será aplicado em todos os regimes. Para isolar a inicialização nos experimentos, **scratch e modelos pré-treinados receberão a mesma normalização nesta L06**.

In [ ]:
PESOS_PRETREINADOS = ResNet18_Weights.DEFAULT
transform_oficial = PESOS_PRETREINADOS.transforms()
MEDIA_PRETREINO = list(transform_oficial.mean)
DESVIO_PRETREINO = list(transform_oficial.std)

print("Média do pré-treino:", MEDIA_PRETREINO)
print("Desvio do pré-treino:", DESVIO_PRETREINO)

operacoes_treino = [transforms.Resize(TAMANHO_IMAGEM)]
if USAR_GEOMETRICO:
    operacoes_treino += [
        transforms.RandomRotation(15),
        transforms.RandomHorizontalFlip(p=0.5),
    ]
if USAR_COR:
    operacoes_treino.append(
        transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.10)
    )
if USAR_DISTORCAO:
    operacoes_treino.append(
        transforms.RandomAffine(
            degrees=0,
            translate=(0.08, 0.08),
            scale=(0.90, 1.10),
            shear=8,
        )
    )
operacoes_treino += [
    transforms.ToTensor(),
    transforms.Normalize(MEDIA_PRETREINO, DESVIO_PRETREINO),
]

transform_treino = transforms.Compose(operacoes_treino)
transform_val_teste = transforms.Compose([
    transforms.Resize(TAMANHO_IMAGEM),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA_PRETREINO, DESVIO_PRETREINO),
])

print("Transform de treino:")
print(transform_treino)
print("\nTransform de validação/teste:")
print(transform_val_teste)

In [ ]:
dataset_treino = datasets.ImageFolder(base_dir / "train", transform=transform_treino)
dataset_val = datasets.ImageFolder(base_dir / "val", transform=transform_val_teste)
dataset_teste = datasets.ImageFolder(base_dir / "test", transform=transform_val_teste)

loader_treino = DataLoader(dataset_treino, batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
loader_teste = DataLoader(dataset_teste, batch_size=BATCH_SIZE, shuffle=False)

print("Classes:", dataset_treino.classes)
print("Treino:", len(dataset_treino), "| Validação:", len(dataset_val), "| Teste:", len(dataset_teste))

### 2.1 A função de perda continua precisando de pesos por classe?

A L05 mostrou `CrossEntropyLoss(weight=...)`. Antes de reaproveitar isso automaticamente, vamos verificar se existe desequilíbrio real no conjunto principal.

In [ ]:
contagens = pd.Series(dataset_treino.targets).value_counts().sort_index()
tabela_classes = pd.DataFrame({
    "classe": [dataset_treino.classes[i] for i in contagens.index],
    "quantidade": contagens.values,
})

pesos_inversos = len(dataset_treino) / (len(dataset_treino.classes) * contagens.values)
tabela_classes["peso_inverso_sugerido"] = pesos_inversos

display(tabela_classes)
print("Nesta lição, o treino principal usa CrossEntropyLoss() sem pesos, porque o dataset principal já foi balanceado.")
criterio = nn.CrossEntropyLoss()

## 3. Construindo as quatro versões

O diagrama abaixo mostra como o descongelamento aumenta progressivamente.

![Descongelamento progressivo](https://drive.google.com/thumbnail?id=1tZwIdMpRuImEFYQBLwhYTD9ACzXTGXGE&sz=w1600)


A célula a seguir é um **exemplo guiado**. Ela centraliza a configuração para que a única diferença entre os modos seja explícita.

In [ ]:
MODOS_VALIDOS = ["scratch", "ft_classificador", "ft_parcial", "ft_completo"]

def criar_resnet18_modo(modo, p_dropout=0.0, n_classes=3):
    if modo not in MODOS_VALIDOS:
        raise ValueError(f"modo deve ser um de {MODOS_VALIDOS}")

    if modo == "scratch":
        modelo = resnet18(weights=None)
    else:
        modelo = resnet18(weights=PESOS_PRETREINADOS)
        # Começamos congelando tudo; depois destravamos apenas o que o modo pede.
        for parametro in modelo.parameters():
            parametro.requires_grad = False

    n_entradas = modelo.fc.in_features
    modelo.fc = nn.Sequential(
        nn.Dropout(p=p_dropout),
        nn.Linear(n_entradas, n_classes),
    )

    if modo == "scratch":
        for parametro in modelo.parameters():
            parametro.requires_grad = True
    elif modo == "ft_classificador":
        pass  # apenas a nova fc já está treinável
    elif modo == "ft_parcial":
        for parametro in modelo.layer4.parameters():
            parametro.requires_grad = True
    elif modo == "ft_completo":
        for parametro in modelo.parameters():
            parametro.requires_grad = True

    return modelo.to(device)


def contar_parametros(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    treinaveis = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    return total, treinaveis

linhas_parametros = []
for modo in MODOS_VALIDOS:
    modelo = criar_resnet18_modo(modo)
    total, treinaveis = contar_parametros(modelo)
    linhas_parametros.append({
        "modo": modo,
        "parametros_totais": total,
        "parametros_treinaveis": treinaveis,
        "percentual_treinavel": 100 * treinaveis / total,
    })

tabela_parametros = pd.DataFrame(linhas_parametros)
display(tabela_parametros)

### Exercício 1 — O congelamento ficou visível nos números?

Use **a tabela produzida acima**.

1. Ordene mentalmente os quatro regimes do menor para o maior número de parâmetros treináveis.
2. Qual regime preserva a maior parte do backbone? Cite o `percentual_treinavel` que apareceu na sua execução.
3. `scratch` e `ft_completo` possuem praticamente a mesma quantidade de parâmetros treináveis. Por que ainda são experimentos diferentes?

**Sua resposta:**

*(escreva aqui, citando pelo menos dois valores da tabela)*

## 4. Experimento diagnóstico — mesmos hiperparâmetros para pontos de partida diferentes

Agora comparamos dois pontos de partida sob os mesmos hiperparâmetros.

![Experimento com mesmos hiperparâmetros](https://drive.google.com/thumbnail?id=1SCNt1lHqS7cYn0eSSGCfqA2W8qJixrtY&sz=w1600)

As duas redes terão **mesma seed, mesmos loaders, Adam, mesma LR e mesmas épocas**. Uma começa do zero; a outra começa pré-treinada e totalmente destravada.

> Não espere um resultado numérico obrigatório. A pergunta é: o que aconteceu nesta execução e quanto o backbone pré-treinado se afastou do ponto inicial?

In [ ]:
def configurar_modo_treino(modelo, modo):
    # Evita atualizar estatísticas internas dos blocos congelados.
    if modo == "ft_classificador":
        modelo.eval()
        modelo.fc.train()
    elif modo == "ft_parcial":
        modelo.eval()
        modelo.layer4.train()
        modelo.fc.train()
    else:
        modelo.train()


def avaliar_validacao(modelo, loader):
    modelo.eval()
    perda_total = 0.0
    acertos = 0
    total = 0
    with torch.no_grad():
        for imagens, rotulos in loader:
            imagens, rotulos = imagens.to(device), rotulos.to(device)
            logits = modelo(imagens)
            perda = criterio(logits, rotulos)
            perda_total += perda.item() * imagens.size(0)
            acertos += (logits.argmax(dim=1) == rotulos).sum().item()
            total += imagens.size(0)
    return perda_total / total, acertos / total


def treinar_modelo(modelo, modo, lr, epocas=2, nome="modelo"):
    # Reinicia a aleatoriedade do loader/augmentation para comparações mais controladas.
    random.seed(SEMENTE + 100)
    torch.manual_seed(SEMENTE + 100)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEMENTE + 100)

    parametros = [p for p in modelo.parameters() if p.requires_grad]
    otimizador = optim.Adam(parametros, lr=lr)
    historico = []

    for epoca in range(1, epocas + 1):
        configurar_modo_treino(modelo, modo)
        perda_total = 0.0
        acertos = 0
        total = 0

        for imagens, rotulos in loader_treino:
            imagens, rotulos = imagens.to(device), rotulos.to(device)
            otimizador.zero_grad()
            logits = modelo(imagens)
            perda = criterio(logits, rotulos)
            perda.backward()
            otimizador.step()

            perda_total += perda.item() * imagens.size(0)
            acertos += (logits.argmax(dim=1) == rotulos).sum().item()
            total += imagens.size(0)

        perda_val, acc_val = avaliar_validacao(modelo, loader_val)
        linha = {
            "nome": nome,
            "modo": modo,
            "epoca": epoca,
            "lr": lr,
            "perda_treino": perda_total / total,
            "acc_treino": acertos / total,
            "perda_val": perda_val,
            "acc_val": acc_val,
        }
        historico.append(linha)
        print(linha)

    return modelo, pd.DataFrame(historico)


def mudanca_relativa(peso_antes, peso_depois):
    return ((peso_depois - peso_antes).norm() / (peso_antes.norm() + 1e-12)).item()

In [ ]:
LR_DIAGNOSTICO = 5e-3
EPOCAS_DIAGNOSTICO = 10

# Scratch
random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
modelo_diag_scratch = criar_resnet18_modo("scratch", p_dropout=0.0)
modelo_diag_scratch, hist_diag_scratch = treinar_modelo(
    modelo_diag_scratch,
    modo="scratch",
    lr=LR_DIAGNOSTICO,
    epocas=EPOCAS_DIAGNOSTICO,
    nome="scratch_mesmos_hiperparametros",
)

# FT completo ingênuo: mesmos hiperparâmetros do scratch acima
random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
modelo_diag_pre = criar_resnet18_modo("ft_completo", p_dropout=0.0)
conv1_antes = modelo_diag_pre.conv1.weight.detach().clone()
modelo_diag_pre, hist_diag_pre = treinar_modelo(
    modelo_diag_pre,
    modo="ft_completo",
    lr=LR_DIAGNOSTICO,
    epocas=EPOCAS_DIAGNOSTICO,
    nome="ft_completo_ingeno_mesmos_hiperparametros",
)
conv1_depois = modelo_diag_pre.conv1.weight.detach().clone()
variacao_conv1_diag = mudanca_relativa(conv1_antes, conv1_depois)

comparacao_diagnostico = pd.DataFrame([
    {
        "modelo": "scratch",
        "acc_val_final": hist_diag_scratch.iloc[-1]["acc_val"],
        "lr": LR_DIAGNOSTICO,
    },
    {
        "modelo": "pretreinado_destravado",
        "acc_val_final": hist_diag_pre.iloc[-1]["acc_val"],
        "lr": LR_DIAGNOSTICO,
    },
])

display(comparacao_diagnostico)
print("Mudança relativa de conv1 no modelo pré-treinado destravado:", variacao_conv1_diag)

### Exercício 2 — O que o experimento realmente mostrou?

Responda **somente com base nos outputs acima**:

1. Qual das duas redes teve maior `acc_val_final` nesta execução? Cite os dois valores.
2. Quanto `conv1.weight` mudou relativamente no modelo pré-treinado?
3. Os resultados autorizam a frase “FT completo sempre piora o modelo”? Por quê?
4. O que conseguimos afirmar com segurança sobre usar os mesmos hiperparâmetros sem considerar o ponto de partida?

**Sua resposta:**

*(escreva aqui)*

## 5. Comparação curta dos quatro regimes

Agora vamos recriar os modelos do zero e usar uma configuração **apropriada ao regime**. `p=0.0` fica fixo para não misturar dropout com a comparação de congelamento.

Estas taxas são didáticas, não resultado de uma busca de hiperparâmetros:

```text
scratch          → 1e-4
ft_classificador → 1e-3
ft_parcial       → 1e-4
ft_completo      → 1e-5
```

Todos usam a mesma validação e o teste continua fechado.

In [ ]:
LRS_REGIMES = {
    "scratch": 1e-4,
    "ft_classificador": 1e-3,
    "ft_parcial": 1e-4,
    "ft_completo": 1e-5,
}
EPOCAS_REGIMES = 10

historicos_regimes = []

for modo in MODOS_VALIDOS:
    print("\n===", modo, "===")
    random.seed(SEMENTE)
    torch.manual_seed(SEMENTE)
    modelo = criar_resnet18_modo(modo, p_dropout=0.0)
    total, treinaveis = contar_parametros(modelo)
    modelo, hist = treinar_modelo(
        modelo,
        modo=modo,
        lr=LRS_REGIMES[modo],
        epocas=EPOCAS_REGIMES,
        nome=modo,
    )
    hist["parametros_treinaveis"] = treinaveis
    historicos_regimes.append(hist)
    del modelo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

historico_regimes = pd.concat(historicos_regimes, ignore_index=True)
resumo_regimes = (
    historico_regimes.groupby(["modo", "lr", "parametros_treinaveis"], as_index=False)["acc_val"]
    .max()
    .rename(columns={"acc_val": "melhor_acc_val"})
    .sort_values("melhor_acc_val", ascending=False)
)
resumo_regimes.to_csv("resultados_regimes_l06.csv", index=False)
display(resumo_regimes)

In [ ]:
plt.figure(figsize=(9, 5))
for modo in MODOS_VALIDOS:
    dados = historico_regimes[historico_regimes["modo"] == modo]
    plt.plot(dados["epoca"], dados["acc_val"], marker="o", label=modo)
plt.xlabel("Época")
plt.ylabel("Acurácia de validação")
plt.title("Quatro regimes — validação")
plt.legend()
plt.show()

### Exercício 3 — Escolha uma estratégia de transferência

Use `resumo_regimes`.

1. Qual foi o melhor regime **pré-treinado** (`ft_classificador`, `ft_parcial` ou `ft_completo`) na validação?
2. Quantos parâmetros treináveis ele tinha?
3. Scratch ficou acima ou abaixo dele nesta execução?
4. Por que o fato de um regime ter mais parâmetros treináveis não garante maior validação?

**Sua resposta:**

*(escreva aqui, citando os valores da tabela)*

In [ ]:
resumo_transfer = resumo_regimes[resumo_regimes["modo"] != "scratch"].copy()
MODO_TRANSFER_ESCOLHIDO = resumo_transfer.iloc[0]["modo"]
print("Regime de transferência escolhido pela validação:", MODO_TRANSFER_ESCOLHIDO)

## 6. Dropout progressivo — rodada 1: `p = 0.0`

Para enxergar apenas o efeito de `p`, usaremos **FT classificador** nas três rodadas. Ele é rápido e mantém o backbone fixo.

![Dropout progressivo](https://drive.google.com/thumbnail?id=1qn44-4pzsZUl-LPuD0q7qPHanUIYLVrr&sz=w1600)

In [ ]:
EPOCAS_DROPOUT = 10
LR_DROPOUT = 1e-3
modelos_dropout = {}
historicos_dropout = {}

random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
modelo_dropout_00 = criar_resnet18_modo("ft_classificador", p_dropout=0.0)
modelo_dropout_00, hist_dropout_00 = treinar_modelo(
    modelo_dropout_00, "ft_classificador", LR_DROPOUT, EPOCAS_DROPOUT, "dropout_0.0"
)
modelos_dropout[0.0] = modelo_dropout_00
historicos_dropout[0.0] = hist_dropout_00

display(hist_dropout_00[["epoca", "acc_treino", "acc_val"]])

**Registro antes da próxima rodada:** qual foi o melhor `acc_val` de `p=0.0`? O treino ficou muito acima da validação?

**Sua anotação:** *(escreva aqui)*

## 6.1 Rodada 2: `p = 0.3`

Agora mude somente a probabilidade de dropout.

In [ ]:
random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
modelo_dropout_03 = criar_resnet18_modo("ft_classificador", p_dropout=0.3)
modelo_dropout_03, hist_dropout_03 = treinar_modelo(
    modelo_dropout_03, "ft_classificador", LR_DROPOUT, EPOCAS_DROPOUT, "dropout_0.3"
)
modelos_dropout[0.3] = modelo_dropout_03
historicos_dropout[0.3] = hist_dropout_03

comparacao_00_03 = pd.DataFrame({
    "p_dropout": [0.0, 0.3],
    "melhor_acc_val": [hist_dropout_00["acc_val"].max(), hist_dropout_03["acc_val"].max()],
})
display(comparacao_00_03)

**Compare antes de continuar:** `p=0.3` aumentou ou reduziu a melhor validação em relação a `p=0.0`? Cite a diferença.

**Sua anotação:** *(escreva aqui)*

## 6.2 Rodada 3: `p = 0.5`

Só agora testamos uma regularização mais forte.

In [ ]:
random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
modelo_dropout_05 = criar_resnet18_modo("ft_classificador", p_dropout=0.5)
modelo_dropout_05, hist_dropout_05 = treinar_modelo(
    modelo_dropout_05, "ft_classificador", LR_DROPOUT, EPOCAS_DROPOUT, "dropout_0.5"
)
modelos_dropout[0.5] = modelo_dropout_05
historicos_dropout[0.5] = hist_dropout_05

resumo_dropout = pd.DataFrame([
    {"p_dropout": p, "melhor_acc_val": hist["acc_val"].max()}
    for p, hist in historicos_dropout.items()
]).sort_values("melhor_acc_val", ascending=False)
resumo_dropout.to_csv("resultados_dropout_l06.csv", index=False)
display(resumo_dropout)

plt.figure(figsize=(8, 5))
for p, hist in historicos_dropout.items():
    plt.plot(hist["epoca"], hist["acc_val"], marker="o", label=f"p={p}")
plt.xlabel("Época")
plt.ylabel("Acurácia de validação")
plt.title("Dropout progressivo — validação")
plt.legend()
plt.show()

### Exercício 4 — Qual `p` você escolhe?

Use os resultados das três rodadas.

1. Qual `p` atingiu a maior validação?
2. O valor mais alto de dropout foi necessariamente o melhor?
3. Em qual rodada a diferença entre `acc_treino` e `acc_val` pareceu maior no final?

**Sua resposta:**

*(escreva aqui, citando valores das tabelas/curvas)*

In [ ]:
P_DROPOUT_ESCOLHIDO = float(resumo_dropout.iloc[0]["p_dropout"])
print("Dropout escolhido pela validação:", P_DROPOUT_ESCOLHIDO)

## 7. Treinando a configuração final

Agora juntamos **o melhor regime pré-treinado da Seção 5** com **o dropout escolhido na Seção 6**.

Somente depois dessa escolha o teste será aberto.

In [ ]:
EPOCAS_FINAL = 10
LR_FINAL = LRS_REGIMES[MODO_TRANSFER_ESCOLHIDO]

random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
modelo_final = criar_resnet18_modo(MODO_TRANSFER_ESCOLHIDO, p_dropout=P_DROPOUT_ESCOLHIDO)
modelo_final, historico_final = treinar_modelo(
    modelo_final,
    modo=MODO_TRANSFER_ESCOLHIDO,
    lr=LR_FINAL,
    epocas=EPOCAS_FINAL,
    nome="resnet18_transfer_l06_final",
)

# Teste: UMA vez, depois da escolha.
perda_teste, acc_teste = avaliar_validacao(modelo_final, loader_teste)
print("Regime final:", MODO_TRANSFER_ESCOLHIDO)
print("Dropout final:", P_DROPOUT_ESCOLHIDO)
print("Acurácia de teste final:", acc_teste)

nome_modelo_final = f"ResNet18IrisTransfer_L06_{MODO_TRANSFER_ESCOLHIDO}_dropout{P_DROPOUT_ESCOLHIDO}"
resultados_modulo = resultados_modulo[
    ~resultados_modulo["modelo"].astype(str).str.startswith("ResNet18IrisTransfer_L06_")
]
nova_linha = pd.DataFrame({"modelo": [nome_modelo_final], "acuracia_teste": [acc_teste]})
resultados_modulo = pd.concat([resultados_modulo, nova_linha], ignore_index=True)
resultados_modulo.to_csv("resultados_modulo_m04.csv", index=False)

torch.save(modelo_final.state_dict(), "resnet18_transfer_l06.pth")
pasta_modulo_drive.mkdir(parents=True, exist_ok=True)
import shutil
for nome in ('resultados_modulo_m04.csv', 'resnet18_transfer_l06.pth'):
    shutil.copy2(nome, pasta_modulo_drive / nome)
print('Resultados e modelo final salvos em:', pasta_modulo_drive)
display(resultados_modulo.tail())

### Comparação com a M04-L05

A tabela da L05 salva o modelo com um nome que começa por `ResNet18IrisMelhorada_L05_`. A célula abaixo procura esse prefixo, em vez de depender de um nome fixo que pode mudar conforme a augmentation vencedora.

In [ ]:
linha_l05 = resultados_modulo[
    resultados_modulo["modelo"].astype(str).str.startswith("ResNet18IrisMelhorada_L05_")
]
display(linha_l05[["modelo", "acuracia_teste"]])
print("L06:", nome_modelo_final, "->", acc_teste)

## 8. Logits → softmax → probabilidades → CSV

![Logits e probabilidades](https://drive.google.com/thumbnail?id=1L9aqRMOqszT49hQOLqTbISnbwFsszykX&sz=w1600)

Os loaders de validação e teste já usam `shuffle=False`. Agora salvamos uma linha por imagem, sempre com o modelo em `eval()`.

In [ ]:
def gerar_predicoes(modelo, dataset, loader):
    modelo.eval()
    caminhos = [caminho for caminho, _ in dataset.samples]
    rotulos_verdadeiros = []
    classes_previstas = []
    probabilidades = []

    with torch.no_grad():
        for imagens, rotulos in loader:
            imagens = imagens.to(device)
            logits = modelo(imagens)
            probs = torch.softmax(logits, dim=1)
            rotulos_verdadeiros.extend(rotulos.tolist())
            classes_previstas.extend(probs.argmax(dim=1).cpu().tolist())
            probabilidades.extend(probs.cpu().tolist())

    tabela = pd.DataFrame({
        "indice_amostra": range(len(caminhos)),
        "caminho_imagem": caminhos,
        "rotulo_verdadeiro": [dataset.classes[r] for r in rotulos_verdadeiros],
        "classe_prevista": [dataset.classes[c] for c in classes_previstas],
    })
    for i, classe in enumerate(dataset.classes):
        tabela[f"probabilidade_{classe}"] = [p[i] for p in probabilidades]
    return tabela

predicoes_val = gerar_predicoes(modelo_final, dataset_val, loader_val)
predicoes_teste = gerar_predicoes(modelo_final, dataset_teste, loader_teste)

predicoes_val.to_csv("predicoes_val_resnet18_transfer_l06.csv", index=False)
predicoes_teste.to_csv("predicoes_teste_resnet18_transfer_l06.csv", index=False)
for nome in ('predicoes_val_resnet18_transfer_l06.csv', 'predicoes_teste_resnet18_transfer_l06.csv'):
    shutil.copy2(nome, pasta_modulo_drive / nome)
print('Predições salvas em:', pasta_modulo_drive)

display(predicoes_teste.head())
colunas_prob = [c for c in predicoes_teste.columns if c.startswith("probabilidade_")]
print("Soma das probabilidades da primeira linha:", predicoes_teste.loc[0, colunas_prob].sum())

## 9. Erro comum — dropout ativo durante a inferência

Vamos usar o modelo treinado com `p=0.5` para tornar o efeito visível. Primeiro colocamos o modelo em avaliação, mas reativamos **apenas a cabeça** em modo de treino. Assim o backbone fica estável e o dropout volta a atuar.

In [ ]:
modelo_erro = modelos_dropout[0.5]
modelo_erro.eval()
modelo_erro.fc.train()  # ERRO DELIBERADO: dropout da cabeça volta a ficar ativo

imagens_exemplo, _ = next(iter(loader_val))
imagens_exemplo = imagens_exemplo.to(device)

with torch.no_grad():
    probs_1 = torch.softmax(modelo_erro(imagens_exemplo), dim=1)
    probs_2 = torch.softmax(modelo_erro(imagens_exemplo), dim=1)

diferenca_dropout_ativo = (probs_1 - probs_2).abs().mean().item()
print("Diferença média com dropout ativo:", diferenca_dropout_ativo)

modelo_erro.eval()  # correção
with torch.no_grad():
    probs_3 = torch.softmax(modelo_erro(imagens_exemplo), dim=1)
    probs_4 = torch.softmax(modelo_erro(imagens_exemplo), dim=1)

diferenca_eval = (probs_3 - probs_4).abs().mean().item()
print("Diferença média após model.eval():", diferenca_eval)

### Exercício 5 — Diagnóstico da inferência

1. Qual foi a diferença média com dropout ativo?
2. Qual foi a diferença após `model.eval()`?
3. Por que salvar probabilidades com dropout ativo seria especialmente problemático para qualquer análise posterior das probabilidades?

**Sua resposta:**

*(escreva aqui usando os dois valores impressos)*

## 10. Exercício final — o mesmo FT completo com uma LR agressiva

Este exercício condensa a ideia central da lição. O FT completo “cuidadoso” da comparação usou `1e-5`. Agora você vai criar **uma nova evidência** mudando apenas a LR para a mesma `LR_DIAGNOSTICO` usada no experimento ingênuo.

O teste continuará fechado. Compare na validação e meça novamente `conv1`.

In [ ]:
# ALTERE apenas esta variável se quiser experimentar outra intensidade.
LR_FT_COMPLETO_FORTE = LR_DIAGNOSTICO

random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
modelo_ft_forte = criar_resnet18_modo("ft_completo", p_dropout=0.0)
conv1_ft_forte_antes = modelo_ft_forte.conv1.weight.detach().clone()
modelo_ft_forte, hist_ft_forte = treinar_modelo(
    modelo_ft_forte,
    modo="ft_completo",
    lr=LR_FT_COMPLETO_FORTE,
    epocas=EPOCAS_REGIMES,
    nome="ft_completo_lr_forte_exercicio_final",
)
conv1_ft_forte_depois = modelo_ft_forte.conv1.weight.detach().clone()
variacao_ft_forte = mudanca_relativa(conv1_ft_forte_antes, conv1_ft_forte_depois)

melhor_val_ft_cuidadoso = resumo_regimes.loc[
    resumo_regimes["modo"] == "ft_completo", "melhor_acc_val"
].iloc[0]
melhor_val_ft_forte = hist_ft_forte["acc_val"].max()

print("FT completo cuidadoso — LR:", LRS_REGIMES["ft_completo"], "melhor val:", melhor_val_ft_cuidadoso)
print("FT completo forte — LR:", LR_FT_COMPLETO_FORTE, "melhor val:", melhor_val_ft_forte)
print("Mudança relativa de conv1 no FT forte:", variacao_ft_forte)

**Perguntas finais (responda depois de executar):**

1. A LR forte melhorou ou piorou a validação em relação ao FT completo cuidadoso? Cite os valores.
2. Quanto `conv1` mudou no FT forte?
3. A sua evidência sustenta a ideia de que “FT completo é errado” ou a ideia de que **a forma de treiná-lo importa**?
4. Se sua acurácia forte tiver ficado boa mesmo assim, que outra evidência do output ainda mostra que o backbone foi alterado de forma mais agressiva?

**Sua resposta:**

*(escreva aqui)*

## 11. Reflexão final

Responda com base no que você executou:

1. Qual foi o melhor regime pré-treinado na validação e quantos parâmetros treináveis ele tinha?
2. O regime com mais parâmetros treináveis foi o melhor? O que isso mostra?
3. Qual `p` de dropout venceu e por quanto?
4. Por que o resultado desta lição não é suficiente para afirmar cientificamente que um regime é universalmente superior?
5. O que os CSVs de probabilidades permitem analisar além da acurácia final?

**Sua resposta:**

*(escreva aqui)*

## 12. Checklist de conclusão

* [ ] Reutilizei `dataset_iris` e a divisão original.
* [ ] Reproduzi minha decisão de augmentation da L05.
* [ ] Usei a normalização dos pesos pré-treinados.
* [ ] Confirmei que o dataset principal está balanceado.
* [ ] Instanciei Scratch, FT classificador, FT parcial e FT completo.
* [ ] Contei parâmetros treináveis dos quatro regimes.
* [ ] Executei o experimento com hiperparâmetros idênticos.
* [ ] Analisei validação e mudança de `conv1`.
* [ ] Comparei os quatro regimes sem usar o teste.
* [ ] Rodei dropout progressivamente em `0.0`, `0.3`, `0.5`.
* [ ] Escolhi regime e dropout pela validação.
* [ ] Avaliei o teste apenas no modelo final.
* [ ] Salvei `resnet18_transfer_l06.pth`.
* [ ] Salvei `resultados_regimes_l06.csv` e `resultados_dropout_l06.csv`.
* [ ] Salvei os CSVs de predições/probabilidades de validação e teste.
* [ ] Corrigi o erro de inferência com dropout ativo.
* [ ] Concluí o exercício final usando nova evidência.

## 13. O que entregar

Mantenha salvos:

* este notebook executado;
* `resultados_regimes_l06.csv`;
* `resultados_dropout_l06.csv`;
* `resultados_modulo_m04.csv` atualizado;
* `resnet18_transfer_l06.pth`;
* `predicoes_val_resnet18_transfer_l06.csv`;
* `predicoes_teste_resnet18_transfer_l06.csv`;
* respostas dos exercícios e reflexão final.
Esta lição encerra o M04. Os resultados, o checkpoint e as probabilidades ficam disponíveis para análise posterior, sem depender de outras lições deste módulo.
